# 1. Setup

In [12]:
# from google.colab import drive
# drive.mount('/content/drive')

In [13]:
import html
import random

# from google.colab import userdata
# import jax
import keras
from keras import layers
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf

import gc
import os
import threading
import time

import psutil

In [14]:
DATASET_PATH = './dataset/'
TRAIN_CSV = DATASET_PATH + 'train.csv'
TEST_CSV = DATASET_PATH + 'test.csv'

TRAINING_SEEDS = [123, 231, 321]
CORRUPTION_SEED = 0
SPLIT_SEED = 1

CORRUPTIONS = ['swap', 'substitution', 'deletion', 'insertion']
CORRUPTION_LEVELS = [0, 5, 10, 15, 20]

STANDARDIZATION = 'lower_and_strip_punctuation'

WORD_MAX_TOKENS = 5000
WORD_SEQUENCE_LENGTH = 69
WORD_EMBEDDING_DIM = 32

CHAR_MAX_TOKENS = 200
CHAR_SEQUENCE_LENGTH = 443     
CHAR_EMBEDDING_DIM = 32

CONV_FILTERS = 128
KERNEL_SIZE = 5
DENSE_UNITS = 64

HIDDEN_ACTIVATION = "relu"
OUTPUT_ACTIVATION = "softmax"

DROPOUT_RATE = 0.2

NUM_CLASSES = 4

LEARNING_RATE = 1e-3
ALPHA = 0.01
DECAY_STEPS = 2820

EPOCHS = 50
PATIENCE = 5
BATCH_SIZE = 64

In [15]:
QWERTY_NEIGHBORS = {
    '`': ['1', 'q'],
    '~': ['!', 'Q'],
    '1': ['`', '2', 'q', 'w'],
    '!': ['~', '@', 'Q', 'W'],
    '2': ['1', '3', 'w', 'e'],
    '@': ['!', '#', 'W', 'E'],
    '3': ['2', '4', 'e', 'r'],
    '#': ['@', '$', 'E', 'R'],
    '4': ['3', '5', 'r', 't'],
    '$': ['#', '%', 'R', 'T'],
    '5': ['4', '6', 't', 'y'],
    '%': ['$', '^', 'T', 'Y'],
    '6': ['5', '7', 'y', 'u'],
    '^': ['%', '&', 'Y', 'U'],
    '7': ['6', '8', 'u', 'i'],
    '&': ['^', '*', 'U', 'I'],
    '8': ['7', '9', 'i', 'o'],
    '*': ['&', '(', 'I', 'O'],
    '9': ['8', '0', 'o', 'p'],
    '(': ['*', ')', 'O', 'P'],
    '0': ['9', '-', 'p', '['],
    ')': ['(', '_', 'P', '{'],
    '-': ['0', '=', '[', ']'],
    '_': [')', '+', '{', '}'],
    '=': ['-', ']', '['],
    '+': ['_', '}', '|'],

    'q': ['`', '1', 'w', 'a'],
    'Q': ['~', '!', 'W', 'A'],
    'w': ['1', '2', 'q', 'e', 'a', 's'],
    'W': ['!', '@', 'Q', 'E', 'A', 'S'],
    'e': ['2', '3', 'w', 'r', 's', 'd'],
    'E': ['@', '#', 'W', 'R', 'S', 'D'],
    'r': ['3', '4', 'e', 't', 'd', 'f'],
    'R': ['#', '$', 'E', 'T', 'D', 'F'],
    't': ['4', '5', 'r', 'y', 'f', 'g'],
    'T': ['$', '%', 'R', 'Y', 'F', 'G'],
    'y': ['5', '6', 't', 'u', 'g', 'h'],
    'Y': ['%', '^', 'T', 'U', 'G', 'H'],
    'u': ['6', '7', 'y', 'i', 'h', 'j'],
    'U': ['^', '&', 'Y', 'I', 'H', 'J'],
    'i': ['7', '8', 'u', 'o', 'j', 'k'],
    'I': ['&', '*', 'U', 'O', 'J', 'K'],
    'o': ['8', '9', 'i', 'p', 'k', 'l'],
    'O': ['*', '(', 'I', 'P', 'K', 'L'],
    'p': ['9', '0', 'o', '[', 'l', ';'],
    'P': ['(', ')', 'O', '{', 'L', ':'],
    '[': ['0', '-', 'p', ']', ';', "'"],
    '{': [')', '_', 'P', '}', ':', '"'],
    ']': ['-', '=', '[', '\\', "'"],
    '}': ['_', '+', '{', '|', '"'],
    '\\': ['=', ']'],
    '|': ['+', '}'],

    'a': ['q', 'w', 's', 'z'],
    'A': ['Q', 'W', 'S', 'Z'],
    's': ['w', 'e', 'a', 'd', 'z', 'x'],
    'S': ['W', 'E', 'A', 'D', 'Z', 'X'],
    'd': ['e', 'r', 's', 'f', 'x', 'c'],
    'D': ['E', 'R', 'S', 'F', 'X', 'C'],
    'f': ['r', 't', 'd', 'g', 'c', 'v'],
    'F': ['R', 'T', 'D', 'G', 'C', 'V'],
    'g': ['t', 'y', 'f', 'h', 'v', 'b'],
    'G': ['T', 'Y', 'F', 'H', 'V', 'B'],
    'h': ['y', 'u', 'g', 'j', 'b', 'n'],
    'H': ['Y', 'U', 'G', 'J', 'B', 'N'],
    'j': ['u', 'i', 'h', 'k', 'n', 'm'],
    'J': ['U', 'I', 'H', 'K', 'N', 'M'],
    'k': ['i', 'o', 'j', 'l', 'm', ','],
    'K': ['I', 'O', 'J', 'L', 'M', '<'],
    'l': ['o', 'p', 'k', ';', ',', '.'],
    'L': ['O', 'P', 'K', ':', '<', '>'],
    ';': ['p', '[', 'l', "'", '.', '/'],
    ':': ['P', '{', 'L', '"', '>', '?'],
    "'": ['[', ']', ';', '/'],
    '"': ['{', '}', ':', '?'],

    'z': ['a', 's', 'x'],
    'Z': ['A', 'S', 'X'],
    'x': ['s', 'd', 'z', 'c'],
    'X': ['S', 'D', 'Z', 'C'],
    'c': ['d', 'f', 'x', 'v'],
    'C': ['D', 'F', 'X', 'V'],
    'v': ['f', 'g', 'c', 'b'],
    'V': ['F', 'G', 'C', 'B'],
    'b': ['g', 'h', 'v', 'n'],
    'B': ['G', 'H', 'V', 'N'],
    'n': ['h', 'j', 'b', 'm'],
    'N': ['H', 'J', 'B', 'M'],
    'm': ['j', 'k', 'n', ','],
    'M': ['J', 'K', 'N', '<'],
    ',': ['k', 'l', 'm', '.'],
    '<': ['K', 'L', 'M', '>'],
    '.': ['l', ';', ',', '/'],
    '>': ['L', ':', '<', '?'],
    '/': [';', "'", '.'],
    '?': [':', '"', '>']
}

In [16]:
tf.config.experimental.enable_op_determinism()
corruption_rng = np.random.default_rng(CORRUPTION_SEED)

In [17]:
# jax.devices('cpu')

In [18]:
# try:
#     print(jax.devices('gpu'))
# except:
#     print('No GPU')

# 2. Data

## 2.1 Loading

In [19]:
train_val_set = pd.read_csv(TRAIN_CSV).groupby('Class Index', sort=False).head(3500)
train_val_set = train_val_set.drop(columns=['Title'])
train_val_set['Description'] = train_val_set['Description'].map(html.unescape)
train_val_set['Description'] = train_val_set['Description'].str.replace('\\', ' ').str.replace('quot;', '').str.replace('--', '-').str.replace(r'\s+', ' ', regex=True).str.strip()
train_val_set

,Class Index,Description
0,3,"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Reuters - Private investment firm Carlyle Grou...
2,3,Reuters - Soaring crude prices plus worries ab...
3,3,Reuters - Authorities have halted oil export f...
4,3,"AFP - Tearaway world oil prices, toppling reco..."
...,...,...
14654,3,No. 7 carrier must reach agreements with union...
14655,3,"China Petroleum amp; Chemical Corp., Asia #39;..."
14661,3,Reuters - U.S. consumer spending rebounded sha...
14662,3,Reuters - U.S. stocks are set to open lower on...


In [20]:
p99_chars = train_val_set['Description'].str.len().quantile(0.99)

p99_words = train_val_set['Description'].str.split().str.len().quantile(0.99)

print('99th percentile characters:', p99_chars)
print('99th percentile words:', p99_words)

99th percentile characters: 443.0
99th percentile words: 69.0


In [21]:
train_df: pd.DataFrame
val_df: pd.DataFrame 

train_df, val_df = train_test_split(train_val_set, test_size=1/7, random_state=SPLIT_SEED, stratify=train_val_set['Class Index'])
val_df['Class Index'].value_counts()

Class Index
4    500
2    500
1    500
3    500
Name: count, dtype: int64

In [22]:
train_df['Class Index'].value_counts()

Class Index
2    3000
4    3000
1    3000
3    3000
Name: count, dtype: int64

In [23]:
test_df = pd.read_csv(TRAIN_CSV).groupby("Class Index", sort=False).head(500).drop(columns=['Title'])
test_df['Description'] = test_df['Description'].map(html.unescape)
test_df['Description'] = test_df['Description'].str.replace('\\', '').str.replace('quot;', '').str.replace('--', '-').str.replace(r'\s+', ' ', regex=True).str.strip()
test_df['Class Index'].value_counts()

Class Index
3    500
4    500
2    500
1    500
Name: count, dtype: int64

In [24]:
test_df

,Class Index,Description
0,3,"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Reuters - Private investment firm Carlyle Grou...
2,3,Reuters - Soaring crude prices plus worriesabo...
3,3,Reuters - Authorities have halted oil exportfl...
4,3,"AFP - Tearaway world oil prices, toppling reco..."
...,...,...
2488,2,"What if the Olympic Games -started here 2,780 ..."
2489,2,More security will be placed around the fields...
2498,2,After an hour of interrogation by a three-man ...
2506,2,"ATHENS, Greece - The night before, Michael Phe..."


## 2.2 Corruption

In [25]:
def apply_corruption(row: str, corruption_level: int) -> str:
    corruption_chance = corruption_level / 100
    words = row.split(' ')
    for k, word in enumerate(words):
        if corruption_rng.random() < corruption_chance:
            chars = list(word)
            i = corruption_rng.integers(len(chars))
            match corruption_rng.choice(CORRUPTIONS):
                case 'swap':
                    if len(chars) > 1:
                        if i < len(chars) - 1 and chars[i+1] != chars[i]:
                            chars[i], chars[i + 1] = chars[i + 1], chars[i]
                        else:
                            chars[i], chars[i - 1] = chars[i - 1], chars[i]
                case 'substitution':
                    chars[i] = corruption_rng.choice(QWERTY_NEIGHBORS[chars[i]])
                case 'deletion':
                    del chars[i]
                case 'insertion':
                    chars.insert(i+1, corruption_rng.choice(QWERTY_NEIGHBORS[chars[i]]))
            words[k] = ''.join(chars)
    return ' '.join(words)

In [26]:
test_sets: dict[int, pd.DataFrame] = {}

for corruption_level in CORRUPTION_LEVELS:
    test_sets[corruption_level] = test_df.assign(Description=test_df['Description'].map(lambda x: apply_corruption(x, corruption_level))) #type: ignore

In [27]:
test_sets

{0:       Class Index                                        Description
 0               3  Reuters - Short-sellers, Wall Street's dwindli...
 1               3  Reuters - Private investment firm Carlyle Grou...
 2               3  Reuters - Soaring crude prices plus worriesabo...
 3               3  Reuters - Authorities have halted oil exportfl...
 4               3  AFP - Tearaway world oil prices, toppling reco...
 ...           ...                                                ...
 2488            2  What if the Olympic Games -started here 2,780 ...
 2489            2  More security will be placed around the fields...
 2498            2  After an hour of interrogation by a three-man ...
 2506            2  ATHENS, Greece - The night before, Michael Phe...
 2507            2  ATHENS, Greece The slogan for these Olympics i...
 
 [2000 rows x 2 columns],
 5:       Class Index                                        Description
 0               3  Reuters - Short-sellers, Wall Street

## 2.3 Vectorization

In [28]:
word_vectorizer = keras.layers.TextVectorization(max_tokens=WORD_MAX_TOKENS, standardize=STANDARDIZATION, split='whitespace', output_mode='int', output_sequence_length=WORD_SEQUENCE_LENGTH)
word_vectorizer.adapt(train_df['Description'])

char_vectorizer = keras.layers.TextVectorization(max_tokens=CHAR_MAX_TOKENS, standardize=STANDARDIZATION, split='character', output_mode='int', output_sequence_length=CHAR_SEQUENCE_LENGTH)
char_vectorizer.adapt(train_df['Description'])

In [29]:
words_train_X = word_vectorizer(train_df['Description'].to_numpy())
words_train_Y = train_df['Class Index'] - 1

words_val_X = word_vectorizer(train_df['Description'].to_numpy())
words_val_Y = train_df['Class Index'] - 1

In [30]:
char_train_X = char_vectorizer(val_df['Description'].to_numpy())
char_train_Y = val_df['Class Index'] - 1

char_val_X = char_vectorizer(val_df['Description'].to_numpy())
char_val_Y = val_df['Class Index'] - 1

# 3. Models

In [31]:
def make_optimizer():
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LEARNING_RATE,
        decay_steps=DECAY_STEPS,
        alpha=ALPHA,
    )

    return keras.optimizers.Adam(
        learning_rate=lr_schedule #type: ignore
    )

In [32]:
def build_word_model():
    model = keras.Sequential([
        keras.Input(
            shape=(WORD_SEQUENCE_LENGTH,),
            dtype='int32',
        ),

        layers.Embedding(
            input_dim=len(word_vectorizer.get_vocabulary()),
            output_dim=WORD_EMBEDDING_DIM,
        ),

        layers.Conv1D(
            filters=CONV_FILTERS,
            kernel_size=KERNEL_SIZE,
            activation=HIDDEN_ACTIVATION,
        ),

        layers.GlobalMaxPooling1D(),

        layers.Dense(
            units=DENSE_UNITS,
            activation=HIDDEN_ACTIVATION,
        ),

        layers.Dense(
            units=NUM_CLASSES,
            activation=OUTPUT_ACTIVATION,
        ),
    ])

    model.compile(
        optimizer=make_optimizer(),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    return model

In [33]:
def build_char_model():
    model = keras.Sequential([
        keras.Input(
            shape=(CHAR_SEQUENCE_LENGTH,),
            dtype='int32',
        ),

        layers.Embedding(
            input_dim=len(char_vectorizer.get_vocabulary()),
            output_dim=CHAR_EMBEDDING_DIM,
        ),

        layers.Conv1D(
            filters=CONV_FILTERS,
            kernel_size=KERNEL_SIZE,
            activation='relu',
        ),

        layers.GlobalMaxPooling1D(),

        layers.Dense(
            units=DENSE_UNITS,
            activation='relu',
        ),

        layers.Dense(
            units=NUM_CLASSES,
            activation='softmax',
        ),
    ])

    model.compile(
        optimizer=make_optimizer(),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    return model

In [34]:
word_model = build_word_model()
char_model = build_char_model()

word_model.summary()
char_model.summary()
print(f'Word model has {(word_model.count_params()/char_model.count_params()):.2f}x as many parameters than the character model')

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 69, 32)         │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 65, 128)        │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 189,124 (738.77 KB)

 Trainable params: 189,124 (738.77 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 443, 32)        │         1,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 439, 128)       │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,372 (118.64 KB)

 Trainable params: 30,372 (118.64 KB)

 Non-trainable params: 0 (0.00 B)

Word model has 6.23x as many parameters than the character model


# 4. Training

In [35]:
class PeakRAMMonitor:
    def __init__(self, sample_interval: float = 0.1):
        self.process = psutil.Process(os.getpid())
        self.sample_interval = sample_interval

        self.initial_bytes = 0
        self.peak_bytes = 0

        self._stop_event = threading.Event()
        self._thread = None

    def _sample(self):
        while not self._stop_event.is_set():
            rss = self.process.memory_info().rss
            self.peak_bytes = max(self.peak_bytes, rss)
            time.sleep(self.sample_interval)

    def start(self):
        self.initial_bytes = self.process.memory_info().rss
        self.peak_bytes = self.initial_bytes

        self._stop_event.clear()
        self._thread = threading.Thread(
            target=self._sample,
            daemon=True
        )
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        self._thread.join()

        self.peak_bytes = max(
            self.peak_bytes,
            self.process.memory_info().rss
        )

        return {
            'initial_ram_mb': self.initial_bytes / 1024**2,
            'peak_ram_mb': self.peak_bytes / 1024**2,
            'ram_increase_mb': (
                self.peak_bytes - self.initial_bytes
            ) / 1024**2,
        }

In [36]:
def train_model(
    model_builder,
    train_X,
    train_Y,
    val_X,
    val_Y,
    seed: int,
):
    keras.backend.clear_session()
    gc.collect()

    keras.utils.set_random_seed(seed)

    model = model_builder()

    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True,
    )

    ram_monitor = PeakRAMMonitor()

    ram_monitor.start()
    start_time = time.perf_counter()

    try:
        history = model.fit(
            train_X,
            train_Y,
            validation_data=(val_X, val_Y),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[early_stopping],
            verbose=1,
        )
    finally:
        training_time = time.perf_counter() - start_time
        ram_stats = ram_monitor.stop()

    stats = {
        'seed': seed,
        'epochs': len(history.history['loss']),
        'training_time_s': training_time,
        'training_time_per_epoch_s': (
            training_time / len(history.history['loss'])
        ),
        **ram_stats,
    }

    return model, history, stats